In [1]:
pip install imbalanced-learn scikit-learn pandas numpy matplotlib seaborn


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek, SMOTEENN


In [3]:
url = "https://raw.githubusercontent.com/AnjulaMehto/Sampling_Assignment/main/Creditcard_data.csv"
df = pd.read_csv(url)

df.head()


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,1
2,1,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [4]:
X = df.drop("Class", axis=1)
y = df["Class"]

print("Original class distribution:")
print(y.value_counts())


Original class distribution:
Class
0    763
1      9
Name: count, dtype: int64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [6]:
samplings = {
    "Sampling1": RandomOverSampler(random_state=42),
    "Sampling2": RandomUnderSampler(random_state=42),
    "Sampling3": SMOTE(random_state=42),
    "Sampling4": SMOTETomek(random_state=42),
    "Sampling5": SMOTEENN(random_state=42)
}


In [7]:
models = {
    "M1": LogisticRegression(max_iter=1000),
    "M2": DecisionTreeClassifier(),
    "M3": RandomForestClassifier(),
    "M4": SVC(),
    "M5": GaussianNB()
}


In [8]:
results = pd.DataFrame(index=models.keys(), columns=samplings.keys())

for s_name, sampler in samplings.items():
    X_res, y_res = sampler.fit_resample(X_train, y_train)

    for m_name, model in models.items():
        model.fit(X_res, y_res)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred) * 100
        results.loc[m_name, s_name] = round(acc, 2)

results


,Sampling1,Sampling2,Sampling3,Sampling4,Sampling5
M1,92.24,69.4,93.97,93.97,94.4
M2,98.71,38.79,98.71,98.71,97.41
M3,99.14,79.31,99.14,99.14,99.14
M4,96.55,83.62,96.55,96.55,95.69
M5,95.26,62.07,95.26,95.26,94.83


In [9]:
print("Accuracy Table (%):")
results


Accuracy Table (%):


,Sampling1,Sampling2,Sampling3,Sampling4,Sampling5
M1,92.24,69.4,93.97,93.97,94.4
M2,98.71,38.79,98.71,98.71,97.41
M3,99.14,79.31,99.14,99.14,99.14
M4,96.55,83.62,96.55,96.55,95.69
M5,95.26,62.07,95.26,95.26,94.83


In [10]:
best_sampling = results.astype(float).idxmax(axis=1)
best_accuracy = results.astype(float).max(axis=1)

summary = pd.DataFrame({
    "Best Sampling Technique": best_sampling,
    "Best Accuracy (%)": best_accuracy
})

summary


,Best Sampling Technique,Best Accuracy (%)
M1,Sampling5,94.40
M2,Sampling1,98.71
M3,Sampling1,99.14
M4,Sampling1,96.55
M5,Sampling1,95.26
